# Computing Eigenvalues & Eigenvectors

Companion notebook for the [Computing Eigenvalues & Eigenvectors wiki page](https://ml-viz.vercel.app/wiki/eigenvalue-computation).

We implement power iteration and the QR algorithm from scratch, verify them
against `np.linalg.eig`, and plot convergence of power iteration.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  2,
})

np.random.seed(0)

## 1 — Worked example: A = [[4, 1], [2, 3]]

Characteristic equation: $\lambda^2 - 7\lambda + 10 = 0 \Rightarrow \lambda = 5, 2$

In [ ]:
A = np.array([[4., 1.], [2., 3.]])

# Verify by numpy
eigenvalues, eigenvectors = np.linalg.eig(A)
idx = np.argsort(eigenvalues)[::-1]   # sort descending
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("Eigenvalues:", eigenvalues.round(6))    # [5., 2.]
print("Eigenvectors (columns):\n", eigenvectors.round(6))

for i in range(2):
    v   = eigenvectors[:, i]
    lam = eigenvalues[i]
    print(f"\nλ={lam:.0f}: Av={A@v:.4f}, λv={lam*v:.4f}, ✓ {np.allclose(A@v, lam*v)}")

## 2 — Power iteration

Repeatedly multiply a random vector by $A$ and re-normalize. Converges to the
dominant eigenvector at rate $|\lambda_2/\lambda_1|^k$.

In [ ]:
def power_iteration(A, n_iter=200, tol=1e-12):
    b = np.random.randn(A.shape[1])
    b /= np.linalg.norm(b)
    rayleigh_history = []
    error_history    = []
    true_lam = max(np.linalg.eigvals(A).real)

    for _ in range(n_iter):
        b_new = A @ b
        lam   = b @ b_new            # Rayleigh quotient
        b_new /= np.linalg.norm(b_new)
        rayleigh_history.append(lam)
        error_history.append(abs(lam - true_lam))
        if np.linalg.norm(b_new - b) < tol:
            break
        b = b_new

    return lam, b, rayleigh_history, error_history

lam, v, rq_hist, err_hist = power_iteration(A)
print(f"Dominant λ: {lam:.6f}  (true: 5)")
# Normalize so first component is positive
v_normalized = v / v[0]
print(f"Dominant v: [{v_normalized[0]:.4f}, {v_normalized[1]:.4f}]  (true: [1, 1])")

## 3 — Power iteration convergence plot

In [ ]:
# Test on several matrices with different eigenvalue gaps
matrices = {
    'A (λ₁/λ₂ = 5/2 = 2.5)':   np.array([[4., 1.], [2., 3.]]),
    'B (λ₁/λ₂ = 10/9 ≈ 1.1)':  np.array([[9.5, 0.5], [0.5, 9.]]),
    'C (λ₁/λ₂ = 100/1 = 100)': np.array([[100., 1.], [1., 1.]]),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#6366f1', '#f97316', '#20d9d2']

for (label, M), col in zip(matrices.items(), colors):
    _, _, rq_hist, err_hist = power_iteration(M, n_iter=40)
    ax1.semilogy(err_hist, color=col, label=label)
    true_lams = sorted(np.linalg.eigvals(M).real, reverse=True)
    ratio = true_lams[1] / true_lams[0]
    theory = [abs(ratio)**k for k in range(len(err_hist))]
    ax1.semilogy(theory, color=col, linestyle='--', alpha=0.4)

ax1.set_xlabel('Iteration')
ax1.set_ylabel('|Rayleigh quotient error|')
ax1.set_title('Power iteration convergence (solid=actual, dashed=theory)')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Eigendecomposition reconstruction error as rank increases
np.random.seed(7)
B = np.random.randn(6, 6)
B = B @ B.T    # symmetric PSD
evals, evecs = np.linalg.eigh(B)
evals = evals[::-1]; evecs = evecs[:, ::-1]

errors = []
for k in range(1, 7):
    B_k = sum(evals[i] * np.outer(evecs[:,i], evecs[:,i]) for i in range(k))
    errors.append(np.linalg.norm(B - B_k, 'fro'))

ax2.bar(range(1, 7), errors, color='#6366f1', alpha=0.8)
ax2.set_xlabel('Rank k of reconstruction')
ax2.set_ylabel('Frobenius reconstruction error')
ax2.set_title('Rank-k eigendecomposition approximation')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 4 — QR algorithm (finding ALL eigenvalues)

The QR algorithm repeatedly decomposes $A_k = Q_k R_k$ and forms $A_{k+1} = R_k Q_k$.
The diagonal of $A_k$ converges to the eigenvalues.

In [ ]:
def qr_algorithm(A, n_iter=100):
    Ak = A.copy().astype(float)
    diag_history = [np.diag(Ak).copy()]
    for _ in range(n_iter):
        Q, R = np.linalg.qr(Ak)
        Ak = R @ Q
        diag_history.append(np.diag(Ak).copy())
    return np.diag(Ak), np.array(diag_history)

A4 = np.array([[4., 3., 2., 1.],
               [3., 5., 2., 1.],
               [2., 2., 6., 1.],
               [1., 1., 1., 7.]], dtype=float)

qr_evals, diag_hist = qr_algorithm(A4, n_iter=60)
np_evals = np.sort(np.linalg.eigh(A4)[0])[::-1]

print("QR algorithm eigenvalues:", qr_evals.round(6))
print("numpy  eigenvalues:      ", np_evals.round(6))

fig, ax = plt.subplots(figsize=(9, 4))
colors4 = ['#6366f1', '#f97316', '#20d9d2', '#ef4444']
for i, col in enumerate(colors4):
    ax.semilogy(abs(diag_hist[:, i] - np_evals[i]) + 1e-16,
                color=col, label=f'λ{i+1}={np_evals[i]:.2f}')
ax.set_xlabel('QR iteration')
ax.set_ylabel('|diagonal - true eigenvalue|')
ax.set_title('QR algorithm convergence')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## ✏️ Your turn

### Exercise 1 — deflated power iteration

After finding the dominant eigenvector $\mathbf{v}_1$, we can find $\mathbf{v}_2$ by
**deflating**: subtracting the rank-1 component $\lambda_1 \mathbf{v}_1 \mathbf{v}_1^\top$
from $A$, then running power iteration on the deflated matrix.
Implement this for $A = [[4,1],[2,3]]$ and recover $\lambda_2 = 2$, $\mathbf{v}_2 = [1,-2]$.

In [ ]:
# TODO(you): deflate A and run power iteration to find the second eigenpair

A = np.array([[4., 1.], [2., 3.]])
lam1, v1, _, _ = power_iteration(A)
v1 = v1 / np.linalg.norm(v1)

# A_deflated = A - lam1 * np.outer(v1, v1)
# lam2, v2, _, _ = power_iteration(A_deflated)
# print(f"λ₂ ≈ {lam2:.4f}  (true: 2)")
# v2_norm = v2 / v2[0]
# print(f"v₂ ≈ [{v2_norm[0]:.3f}, {v2_norm[1]:.3f}]  (true: [1, -2])")

### Exercise 2 — symmetric power iteration

For a symmetric matrix $A = A^\top$, eigenvectors are **orthogonal**.
Use power iteration to find both eigenvectors of
$B = \begin{bmatrix}3&1\\1&2\end{bmatrix}$
and verify $\mathbf{v}_1 \cdot \mathbf{v}_2 = 0$.

<details>
<summary>Solution</summary>

```python
B = np.array([[3., 1.], [1., 2.]])
lam1, v1, _, _ = power_iteration(B)
B_def = B - lam1 * np.outer(v1, v1)
lam2, v2, _, _ = power_iteration(B_def)

print(f"λ₁={lam1:.4f}, λ₂={lam2:.4f}")
print(f"v₁·v₂ = {np.dot(v1, v2):.8f}  (should ≈ 0)")
```
</details>